# Stage 3: serve the adapted model with vLLM

Checks that **vLLM serves the adapted model** (notebook 07's output) with the config that `zubaan export vllm` generates (`dtype half`): vLLM's tokenization of Urdu equals the extended `tokenizer.json`, and the model answers an Urdu completion and an Urdu chat question, once with thinking off and once with thinking allowed (to see what training on plain text did to the `<think>` behavior).

Settings before running:

- Accelerator: **GPU T4 x2** (one GPU is used). **Not P100**: vLLM needs compute capability 7.0 or higher.
- Internet: **on**. No secret is needed.
- Input: **Add Input > Notebooks**: the output of `07_adapt_main_qwen3`.

Takes about 15 to 25 minutes, mostly installing vLLM (its own Python 3.12 environment) and loading the model. The model is stored as bfloat16 and served as float16 (the T4 has no bfloat16).

Send back `vllm_check_adapted.json`.

In [ ]:
# Install zubaan at one exact commit (only its small core is needed here: the vLLM config generator).
COMMIT = "59f2708"
REPO = f"git+https://github.com/Hashir-Aziz123/zubaan.git@{COMMIT}"
!pip install -q --force-reinstall "zubaan @ {REPO}" tokenizers
!zubaan version

In [ ]:
# CHECK 1: the installed code is the commit above, not a cached older copy.
import json
from importlib import metadata

direct = json.loads(metadata.distribution("zubaan").read_text("direct_url.json"))
installed = direct["vcs_info"]["commit_id"]
print("installed commit:", installed)
assert installed.startswith(COMMIT), f"installed {installed}, wanted {COMMIT}"
print("OK: code version verified")

In [ ]:
import subprocess

# CHECK 2: a GPU that vLLM supports. vLLM needs compute capability 7.0 or higher: a Kaggle T4 is 7.5,
# a P100 is 6.0 and does NOT work. Pick "GPU T4 x2" in the notebook settings.
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
except FileNotFoundError:
    raise SystemExit("No GPU in this session (nvidia-smi is missing). Settings > Accelerator > GPU T4 x2, then run again.") from None
assert r.returncode == 0, "no GPU found: turn on a GPU accelerator (T4) in the notebook settings"
print(r.stdout)
name, cap, mem = [x.strip() for x in r.stdout.splitlines()[0].split(",")]
assert float(cap) >= 7.0, f"{name} has compute capability {cap}; vLLM needs 7.0 or higher. Use a T4."
print("OK: GPU", name, "compute capability", cap, mem)

name_gpu = name

In [ ]:
NAME = "qwen3-urdu-adapted"
MAX_MODEL_LEN = 4096

## 1. Find the adapted model

In [ ]:
import json
from pathlib import Path

# The adapted model is the output of notebook 07 (Add Input > Notebooks); zubaan_extension.json says
# trained: true. vLLM serves this Hugging Face folder directly (no GGUF).
markers = sorted(Path("/kaggle/input").rglob("zubaan_extension.json"))
trained = [m.parent for m in markers if json.loads(m.read_text()).get("trained")]
print("trained model folders:", [str(f) for f in trained])
assert len(trained) == 1, f"need exactly one trained model folder, found {len(trained)}: add notebook 07's output"
MODEL_DIR = trained[0]
assert (MODEL_DIR / "tokenizer.json").exists(), "no tokenizer.json in the model folder"
assert (MODEL_DIR / "model.safetensors").exists() or list(MODEL_DIR.glob("model-*.safetensors")), "no weights"
has_template = "chat_template" in (MODEL_DIR / "tokenizer_config.json").read_text() or (MODEL_DIR / "chat_template.jinja").exists()
assert has_template, "no chat template: /v1/chat/completions would fail"
print("OK: adapted model:", MODEL_DIR)

## 2. Install vLLM (isolated environment)

In [ ]:
# vLLM goes into its own Python 3.12 environment so it cannot replace Kaggle's torch. Takes 5 to 10 minutes.
!pip install -q uv
!uv venv --python 3.12 /tmp/vllm-env
!uv pip install --python /tmp/vllm-env/bin/python vllm
VLLM_VERSION = subprocess.run(
    ["/tmp/vllm-env/bin/python", "-c", "import vllm; print(vllm.__version__)"], capture_output=True, text=True
).stdout.strip()
print("vLLM", VLLM_VERSION)
assert VLLM_VERSION, "vLLM did not install"

## 3. Generate the vLLM config and start the server

In [ ]:
from zubaan.export.vllm import write_vllm_files

# The same generator that 'zubaan export vllm' uses, pointed at this session's model path.
WORK = Path("/kaggle/working")
files = write_vllm_files(WORK / "vllm", str(MODEL_DIR), NAME, MAX_MODEL_LEN)
config_text = files["config"].read_text()
print(config_text)
print(files["script"].read_text())
assert "dtype: half" in config_text and "bfloat16" not in config_text.lower()

In [ ]:
import os
import signal
import time
import urllib.request

LOG = WORK / "vllm_server.log"
env = dict(os.environ, PATH="/tmp/vllm-env/bin:" + os.environ["PATH"])
server = subprocess.Popen(
    f"cd {WORK / 'vllm'} && sh serve_vllm.sh > {LOG} 2>&1", shell=True, env=env, start_new_session=True
)


def get(path, body=None):
    req = urllib.request.Request(
        f"http://127.0.0.1:8000{path}",
        data=None if body is None else json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
    )
    return json.load(urllib.request.urlopen(req, timeout=300))


up = False
for _ in range(120):  # up to 20 minutes: model load, memory profiling and CUDA graph capture
    time.sleep(10)
    if server.poll() is not None:
        break
    try:
        up = NAME in json.dumps(get("/v1/models"))
    except Exception:
        up = False
    if up:
        break
print("server up:", up)
if not up:
    print(LOG.read_text()[-3000:])
assert up, "vLLM did not start: see the log above and /kaggle/working/vllm_server.log"

## 4. Check tokenization and generation

In [ ]:
import time

from tokenizers import Tokenizer

from zubaan.export.check import strip_think, urdu_letter_share

# CHECK 3: the server uses the EXTENDED tokenizer: vLLM's ids for Urdu equal those of tokenizer.json.
tok = Tokenizer.from_file(str(MODEL_DIR / "tokenizer.json"))
sentences = [
    "پاکستان کا دارالحکومت",
    "پاکستان ایک خوبصورت ملک ہے۔",
    "کیا آپ ٹھیک ہیں؟ جی ہاں!",
    "اردو میں 123 اور English کا مرکب mixed script ٹیسٹ",
    "The quick brown fox jumps over the lazy dog.",
]
mismatches = []
for s in sentences:
    want = tok.encode(s, add_special_tokens=False).ids
    got = get("/tokenize", {"model": NAME, "prompt": s, "add_special_tokens": False})["tokens"]
    if got != want:
        mismatches.append((s, want, got))
print(f"tokenization: {len(sentences) - len(mismatches)}/{len(sentences)} identical to tokenizer.json")
assert not mismatches, mismatches

# CHECK 4: it generates Urdu. Greedy, so the output is reproducible.
samples = {}
t0 = time.perf_counter()
out = get("/v1/completions", {"model": NAME, "prompt": sentences[0], "max_tokens": 60, "temperature": 0})
samples["completion"] = {"seconds": round(time.perf_counter() - t0, 2), "usage": out["usage"], "text": out["choices"][0]["text"]}
question = "پاکستان کا دارالحکومت کیا ہے؟"  # "What is the capital of Pakistan?"
for name, kwargs in [("chat_no_think", {"enable_thinking": False}), ("chat_thinking", {})]:
    body = {"model": NAME, "messages": [{"role": "user", "content": question}], "max_tokens": 200, "temperature": 0}
    if kwargs:
        body["chat_template_kwargs"] = kwargs
    r = get("/v1/chat/completions", body)
    samples[name] = {"usage": r["usage"], "text": r["choices"][0]["message"]["content"] or "",
                     "finish_reason": r["choices"][0]["finish_reason"]}
for name, s in samples.items():
    s["answer_urdu_letter_share"] = urdu_letter_share(strip_think(s["text"]))
    print(f"--- {name} | Urdu letters: {s['answer_urdu_letter_share']}")
    print(s["text"])
assert samples["completion"]["usage"]["completion_tokens"] > 0
assert (samples["chat_no_think"]["answer_urdu_letter_share"] or 0) > 0.8, "the chat answer is not mostly Urdu script"

result = {
    "passed": True,
    "vllm": VLLM_VERSION,
    "gpu": name_gpu,
    "compute_capability": cap,
    "dtype": "half",
    "model_dir": str(MODEL_DIR),
    "tokenization_identical": len(sentences),
    "samples": samples,
}
(WORK / "vllm_check_adapted.json").write_text(json.dumps(result, ensure_ascii=False, indent=2))
print("OK: vLLM served the adapted model")

## 5. Stop the server

In [ ]:
os.killpg(os.getpgid(server.pid), signal.SIGTERM)
time.sleep(5)
print(LOG.read_text()[-1500:])